In [37]:
!pip install selenium
!pip install webdriver-manager
!pip install beautifulsoup4

import re
import time

from datetime import datetime
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def extract_stock_data(html):
    soup = BeautifulSoup(html, "html.parser")

    # HTMLから文字列を取得
    text = soup.get_text(" ", strip=True)

    # 日付・始値・高値・安値・終値を取得
    pattern = (
        r"(\d{4})/(\d{1,2})/(\d{1,2})"
        r"\s*始値\s*([\d,.]+)"
        r"\s*高値\s*([\d,.]+)"
        r"\s*安値\s*([\d,.]+)"
        r"\s*終値\s*([\d,.]+)"
    )

    match = re.search(pattern, text)

    if match is None:
        return None

    # 日付を「年-月-日」の形式にして0埋め
    year = int(match.group(1))
    month = int(match.group(2))
    day = int(match.group(3))

    date = f"{year:04d}-{month:02d}-{day:02d}"

    open_price = match.group(4)
    high_price = match.group(5)
    low_price = match.group(6)
    close_price = match.group(7)

    return [
        date,
        open_price,
        high_price,
        low_price,
        close_price
    ]


def get_stock_values(driver, url):
    # Webページへアクセス
    driver.get(url)

    # グラフのiframeが表示されるまで待機
    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, 'iframe[title="iframe-chart"]')
        )
    )

    # グラフのiframeを取得
    chart_iframe = driver.find_element(
        By.CSS_SELECTOR,
        'iframe[title="iframe-chart"]'
    )

    # iframe内へ移動
    driver.switch_to.frame(chart_iframe)

    # グラフが表示されるまで待機
    graph = WebDriverWait(driver, 20).until(
        EC.presence_of_element_located(
            (By.CLASS_NAME, "highcharts-container")
        )
    )

    time.sleep(2)

    stock_values = []

    # グラフの横幅を取得
    graph_width = graph.size["width"]

    # グラフ中央へマウスを移動
    actions = ActionChains(driver)
    actions.move_to_element(graph).perform()

    # 中央からグラフ幅の半分だけ右へ移動
    actions.move_by_offset(
        graph_width // 2 - 1,
        0
    ).perform()

    time.sleep(0.2)

    previous_date = None

    # グラフ右端から1pxずつ左へ移動
    for i in range(graph_width):

        html = driver.page_source

        # extract_stock_data関数でデータを取得
        stock_data = extract_stock_data(html)

        if stock_data is not None:
            # 同じ日付を重複して追加しない
            if stock_data[0] != previous_date:
                stock_values.append(stock_data)
                previous_date = stock_data[0]

        # 1px左へ移動
        ActionChains(driver).move_by_offset(
            -1,
            0
        ).perform()

        time.sleep(0.01)

    # iframeの外へ戻る
    driver.switch_to.default_content()

    return stock_values


# 日経平均株価のURL
url = "https://www.nikkei.com/marketdata/quote/NK225/?type=6month"

# Chromeの設定
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

# Chromeを起動
driver = webdriver.Chrome(options=options)

try:
    # スクレイピング開始時間
    start_time = datetime.now()

    print("スクレイピング開始時間：", start_time)

    # 株価データを取得
    stock_values = get_stock_values(driver, url)

    # スクレイピング終了時間
    end_time = datetime.now()

    print("スクレイピング終了時間：", end_time)

    # かかった時間を計算
    elapsed_time = end_time - start_time

    print("スクレイピングにかかった時間：", elapsed_time)

    print()

    # 日付、始値、高値、安値、終値の順で表示
    for data in stock_values:
        print(
            data[0],
            data[1],
            data[2],
            data[3],
            data[4]
        )

finally:
    # Chromeを終了
    driver.quit()

スクレイピング開始時間： 2026-09-21 10:06:49.531793
スクレイピング終了時間： 2026-09-21 10:10:31.203547
スクレイピングにかかった時間： 0:03:41.671754

2026-09-18 64681.55 65436.57 64403.85 65018.95
2026-09-17 64643.58 64643.58 63824.56 64136.25
2026-09-16 63672.13 63923 63209.92 63923
2026-09-15 63190.37 64101 63067.18 63484.1
2026-09-14 63659.53 63691.53 62726.18 63492.99
2026-09-11 64276.82 64312.02 63208.63 64011.34
2026-09-10 64768.53 65270.95 64186.68 65270.95
2026-09-09 65087.22 65794.03 65017.68 65142.78
2026-09-08 65843.69 66791.84 65269.33 65269.33
2026-09-07 65600.42 66668.71 65600.42 66399.84
2026-09-04 64498.94 65182.45 64228.47 65020.94
2026-09-03 64724.81 64724.81 63772.8 64214.48
2026-09-02 65195.43 65195.43 64215.47 64325.64
2026-09-01 65885.47 66525.7 65576.61 66215.34
2026-08-31 65668.51 66311.93 64832.1 66311.93
2026-08-28 66104.49 66842.82 66012.2 66405.56
2026-08-27 66775.78 66954.69 65782.59 66131.98
2026-08-26 65604.49 66442.34 65390.55 66262.16
2026-08-25 65195.26 65904.24 64609.47 65856.43
2026-08-2